In [ ]:
# Cell 1 — Install tokenizer
# Install the tokenizer library
!pip -q install tiktoken tqdm

In [ ]:
# Cell 2 — Mount Google Drive and set paths
# Mount Google Drive and define paths for both datasets

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/final_project")
OUTPUT_DIR = BASE_DIR / "idea_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# HotpotQA paths
INPUT_PATH = BASE_DIR / "hotpotqa_dev_2017wiki_1000_converted.json"
OUTPUT_PATH = OUTPUT_DIR / "hotpotqa_docs_chunks.json"

# 2WikiMultihopQA paths
TWOWIKI_INPUT_PATH = BASE_DIR / "2wikimultihopqa_dev_2020wiki_1000_converted.json"
TWOWIKI_OUTPUT_PATH = OUTPUT_DIR / "2wikimultihopqa_docs_chunks.json"

print("HotpotQA input file:", INPUT_PATH)
print("HotpotQA output file:", OUTPUT_PATH)

print("2WikiMultihopQA input file:", TWOWIKI_INPUT_PATH)
print("2WikiMultihopQA output file:", TWOWIKI_OUTPUT_PATH)

Mounted at /content/drive
HotpotQA input file: /content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json
HotpotQA output file: /content/drive/MyDrive/final_project/idea_1/hotpotqa_docs_chunks.json
2WikiMultihopQA input file: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json
2WikiMultihopQA output file: /content/drive/MyDrive/final_project/idea_1/2wikimultihopqa_docs_chunks.json


In [ ]:
# Cell 3 — Imports and configuration
# Import required libraries and define chunking configuration
import json
import math
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
import tiktoken

TOKENIZER_NAME = "cl100k_base"

# Hard limits
VALID_MIN_CHUNK_TOKENS = 400
PREFERRED_MIN_CHUNK_TOKENS = 450
MAX_CHUNK_TOKENS = 512

# Preferred target: close to upper bound, but still safely below 512
TARGET_CHUNK_TOKENS = 506

# Long text overlap
LONG_TEXT_OVERLAP = 50

# Very short paragraphs are usually section titles
SHORT_PARAGRAPH_TOKENS = 10

enc = tiktoken.get_encoding(TOKENIZER_NAME)

def count_tokens(text: str) -> int:
    # Count tokens using the selected tokenizer
    return len(enc.encode(text))

In [ ]:
# Cell 4 — Load the dataset
# Load the converted HotpotQA JSON file
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Support both list-based and dict-based JSON structures
if isinstance(raw_data, list):
    examples = raw_data
elif isinstance(raw_data, dict):
    if "examples" in raw_data:
        examples = raw_data["examples"]
    elif "data" in raw_data:
        examples = raw_data["data"]
    else:
        raise ValueError("JSON is a dict, but no 'examples' or 'data' key was found.")
else:
    raise TypeError("Unsupported JSON structure.")

print("Number of examples:", len(examples))
print("Keys in first example:", list(examples[0].keys()))

Number of examples: 1000
Keys in first example: ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']


In [7]:
# Cell 4.5 — Load 2WikiMultihopQA dataset
# Load the converted 2WikiMultihopQA JSON file before chunking

with open(TWOWIKI_INPUT_PATH, "r", encoding="utf-8") as f:
    twowiki_raw_data = json.load(f)

# Support list-based and dict-based JSON structures
if isinstance(twowiki_raw_data, list):
    twowiki_examples = twowiki_raw_data
elif isinstance(twowiki_raw_data, dict):
    if "examples" in twowiki_raw_data:
        twowiki_examples = twowiki_raw_data["examples"]
    elif "converted" in twowiki_raw_data:
        twowiki_examples = twowiki_raw_data["converted"]
    elif "data" in twowiki_raw_data:
        twowiki_examples = twowiki_raw_data["data"]
    else:
        raise ValueError("JSON is a dict, but no 'examples', 'converted', or 'data' key was found.")
else:
    raise TypeError("Unsupported JSON structure.")

print("Number of 2WikiMultihopQA examples:", len(twowiki_examples))
print("Keys in first 2WikiMultihopQA example:", list(twowiki_examples[0].keys()))

Number of 2WikiMultihopQA examples: 1000
Keys in first 2WikiMultihopQA example: ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']


In [8]:
# Cell 5 — Paragraph preparation functions
# Split docs into paragraphs and merge very short paragraphs with the next one

def split_non_empty_paragraphs(doc_text: str):
    """
    Split one Wikipedia document into non-empty paragraphs.
    Paragraph IDs start from 1 inside each title.
    """
    raw_paragraphs = doc_text.split("\n")
    paragraphs = []

    paragraph_id = 1
    for p in raw_paragraphs:
        p = p.strip()
        if not p:
            continue

        paragraphs.append({
            "paragraph_ids": [paragraph_id],
            "text": p,
            "token_count": count_tokens(p)
        })
        paragraph_id += 1

    return paragraphs


def merge_short_paragraphs_with_next(paragraphs):
    """
    Any paragraph with fewer than SHORT_PARAGRAPH_TOKENS tokens is attached to the next paragraph.
    If such a paragraph appears at the end, it is attached to the previous unit when possible.
    """
    merged_units = []
    short_buffer = []

    for para in paragraphs:
        if para["token_count"] < SHORT_PARAGRAPH_TOKENS:
            short_buffer.append(para)
            continue

        if short_buffer:
            combined_ids = []
            combined_texts = []

            for sp in short_buffer:
                combined_ids.extend(sp["paragraph_ids"])
                combined_texts.append(sp["text"])

            combined_ids.extend(para["paragraph_ids"])
            combined_texts.append(para["text"])

            combined_text = "\n".join(combined_texts)

            merged_units.append({
                "paragraph_ids": combined_ids,
                "text": combined_text,
                "token_count": count_tokens(combined_text)
            })

            short_buffer = []
        else:
            merged_units.append(para)

    # Attach trailing short paragraphs to the previous unit if possible
    if short_buffer:
        combined_ids = []
        combined_texts = []

        for sp in short_buffer:
            combined_ids.extend(sp["paragraph_ids"])
            combined_texts.append(sp["text"])

        trailing_text = "\n".join(combined_texts)

        if merged_units:
            merged_units[-1]["paragraph_ids"].extend(combined_ids)
            merged_units[-1]["text"] = merged_units[-1]["text"] + "\n" + trailing_text
            merged_units[-1]["token_count"] = count_tokens(merged_units[-1]["text"])
        else:
            merged_units.append({
                "paragraph_ids": combined_ids,
                "text": trailing_text,
                "token_count": count_tokens(trailing_text)
            })

    return merged_units

In [9]:
# Cell 6 — Long paragraph splitting with overlap
# Split long paragraphs into token windows with overlap

def split_long_text_with_overlap(text, max_tokens=MAX_CHUNK_TOKENS, overlap=LONG_TEXT_OVERLAP):
    """
    Split a text longer than max_tokens into token-based chunks.
    Chunks use token overlap.
    For some lengths, chunks below 400 tokens are unavoidable, so the split is balanced.
    """
    token_ids = enc.encode(text)
    n = len(token_ids)

    if n <= max_tokens:
        return [text]

    # Minimum number of chunks needed with the chosen max size and overlap
    k = math.ceil((n - overlap) / (max_tokens - overlap))
    k = max(k, 2)

    # Balanced window size, including overlap duplication
    window_size = math.ceil((n + overlap * (k - 1)) / k)

    while window_size > max_tokens:
        k += 1
        window_size = math.ceil((n + overlap * (k - 1)) / k)

    chunks = []
    start = 0

    for i in range(k):
        end = min(start + window_size, n)

        if i == k - 1:
            end = n

        piece_ids = token_ids[start:end]
        piece_text = enc.decode(piece_ids)
        chunks.append(piece_text)

        if end >= n:
            break

        start = end - overlap

    return chunks

In [10]:
# Cell 7 — Optimized paragraph grouping per title
# Optimize paragraph grouping inside each title without crossing title boundaries

def add_score(a, b):
    # Add two lexicographic score tuples
    return tuple(x + y for x, y in zip(a, b))


def score_chunk(token_count):
    """
    Score one candidate chunk.

    Lower is better.

    Priority:
    1. Avoid chunks below 400.
    2. Avoid chunks below 450.
    3. Avoid chunks below 500.
    4. Prefer chunks closer to TARGET_CHUNK_TOKENS.
    5. Prefer fewer chunks only after quality is considered.
    """
    below_400 = 1 if token_count < VALID_MIN_CHUNK_TOKENS else 0
    below_450 = 1 if token_count < PREFERRED_MIN_CHUNK_TOKENS else 0
    below_500 = 1 if token_count < 500 else 0

    shortfall_400 = max(0, VALID_MIN_CHUNK_TOKENS - token_count)
    shortfall_450 = max(0, PREFERRED_MIN_CHUNK_TOKENS - token_count)
    shortfall_500 = max(0, 500 - token_count)

    target_deviation = abs(TARGET_CHUNK_TOKENS - token_count)

    return (
        below_400,
        below_450,
        below_500,
        shortfall_400,
        shortfall_450,
        shortfall_500,
        target_deviation,
        1
    )


def chunk_normal_units_dp(units):
    """
    Group consecutive paragraph units into chunks.

    No overlap is used here.
    The algorithm tries to make chunks as close as possible to 500-512 tokens,
    while strongly avoiding chunks below 400 and preferably below 450.
    """
    n = len(units)

    if n == 0:
        return []

    # dp[i] = best score for units[:i]
    dp = [None] * (n + 1)
    prev = [None] * (n + 1)

    zero_score = (0, 0, 0, 0, 0, 0, 0, 0)
    dp[0] = zero_score

    for end in range(1, n + 1):
        best_score = None
        best_start = None

        candidate_texts = []

        for start in range(end - 1, -1, -1):
            candidate_texts.insert(0, units[start]["text"])
            candidate_text = "\n".join(candidate_texts)
            candidate_tokens = count_tokens(candidate_text)

            # Normal paragraph-combination chunks must not exceed 512 tokens
            if candidate_tokens > MAX_CHUNK_TOKENS:
                break

            if dp[start] is None:
                continue

            candidate_score = add_score(dp[start], score_chunk(candidate_tokens))

            if best_score is None or candidate_score < best_score:
                best_score = candidate_score
                best_start = start

        # Fallback should almost never happen unless one unit is already too large
        if best_score is None:
            candidate_text = units[end - 1]["text"]
            candidate_tokens = count_tokens(candidate_text)

            best_score = add_score(dp[end - 1], score_chunk(candidate_tokens))
            best_start = end - 1

        dp[end] = best_score
        prev[end] = best_start

    # Reconstruct chunks
    chunks = []
    idx = n

    while idx > 0:
        start = prev[idx]
        group = units[start:idx]

        paragraph_ids = []
        texts = []

        for u in group:
            paragraph_ids.extend(u["paragraph_ids"])
            texts.append(u["text"])

        chunk_text = "\n".join(texts)

        chunks.append({
            "paragraph_ids": paragraph_ids,
            "text": chunk_text,
            "token_count": count_tokens(chunk_text)
        })

        idx = start

    chunks.reverse()
    return chunks


def chunk_units_for_one_title(units):
    """
    Chunk one title only.

    Long units above 512 tokens are split separately with overlap.
    Normal units are grouped with dynamic programming.
    """
    final_chunks = []
    buffer_units = []

    for unit in units:
        if unit["token_count"] > MAX_CHUNK_TOKENS:
            # Flush normal paragraph buffer first
            if buffer_units:
                final_chunks.extend(chunk_normal_units_dp(buffer_units))
                buffer_units = []

            # Split long text with token overlap
            long_pieces = split_long_text_with_overlap(unit["text"])

            for piece_text in long_pieces:
                final_chunks.append({
                    "paragraph_ids": unit["paragraph_ids"],
                    "text": piece_text,
                    "token_count": count_tokens(piece_text)
                })
        else:
            buffer_units.append(unit)

    if buffer_units:
        final_chunks.extend(chunk_normal_units_dp(buffer_units))

    return final_chunks

In [11]:
# Cell 7.5 — Deduplication helper functions
# Define helper functions for per-dataset title+doc deduplication

from collections import Counter

def normalize_title_for_dedup(title: str) -> str:
    # Normalize title for exact duplicate detection
    return str(title).strip()


def normalize_doc_for_dedup(doc_text: str) -> str:
    # Normalize document text while preserving paragraph order
    lines = str(doc_text).replace("\r\n", "\n").replace("\r", "\n").split("\n")
    lines = [line.strip() for line in lines]
    lines = [line for line in lines if line]
    return "\n".join(lines)


def make_doc_key(title: str, doc_text: str):
    # Deduplication key: same title and same full document text
    return (
        normalize_title_for_dedup(title),
        normalize_doc_for_dedup(doc_text)
    )


def collect_doc_keys_from_examples(examples_list, dataset_name):
    # Collect title+doc keys for duplicate analysis
    records = []

    for example_idx, example in enumerate(examples_list):
        titles = example.get("titles", [])
        docs = example.get("docs", [])

        if not isinstance(titles, list):
            titles = []

        if not isinstance(docs, list):
            continue

        for doc_idx, doc_text in enumerate(docs):
            if not isinstance(doc_text, str) or not doc_text.strip():
                continue

            title = titles[doc_idx] if doc_idx < len(titles) else f"untitled_doc_{doc_idx + 1}"
            normalized_title, normalized_doc = make_doc_key(title, doc_text)

            records.append({
                "dataset": dataset_name,
                "example_idx": example_idx,
                "doc_idx": doc_idx,
                "title": normalized_title,
                "doc_key": (normalized_title, normalized_doc),
                "doc_token_count": count_tokens(normalized_doc),
                "doc_char_count": len(normalized_doc),
                "text_preview": normalized_doc[:500]
            })

    return records

In [12]:
# Cell 7.6 — Analyze HotpotQA duplicate docs
# Check duplicate title+doc pairs inside HotpotQA only

hotpot_doc_records = collect_doc_keys_from_examples(examples, "hotpotqa")
hotpot_doc_keys = [r["doc_key"] for r in hotpot_doc_records]

hotpot_key_counter = Counter(hotpot_doc_keys)
hotpot_unique_keys = set(hotpot_doc_keys)

hotpot_internal_duplicate_count = sum(
    count - 1 for count in hotpot_key_counter.values() if count > 1
)

print("HotpotQA total docs:", len(hotpot_doc_keys))
print("HotpotQA unique title+doc pairs:", len(hotpot_unique_keys))
print("HotpotQA duplicate docs inside HotpotQA:", hotpot_internal_duplicate_count)

hotpot_duplicate_samples = []

for r in hotpot_doc_records:
    if hotpot_key_counter[r["doc_key"]] > 1:
        hotpot_duplicate_samples.append({
            "example_idx": r["example_idx"],
            "doc_idx": r["doc_idx"],
            "title": r["title"],
            "doc_token_count": r["doc_token_count"],
            "text_preview": r["text_preview"]
        })

    if len(hotpot_duplicate_samples) >= 10:
        break

if hotpot_duplicate_samples:
    print("\nSample duplicate title+doc pairs inside HotpotQA:")
    display(pd.DataFrame(hotpot_duplicate_samples))
else:
    print("\nNo duplicate title+doc pairs found inside HotpotQA.")

HotpotQA total docs: 10000
HotpotQA unique title+doc pairs: 9810
HotpotQA duplicate docs inside HotpotQA: 190

Sample duplicate title+doc pairs inside HotpotQA:


,example_idx,doc_idx,title,doc_token_count,text_preview
0,5,3,Loan modification in the United States,5188,Loan modification is the systematic alteration...
1,8,4,Deadfall (1993 film),212,Deadfall is a 1993 crime drama film directed b...
2,14,6,Force India VJM04,315,The Force India VJM04 is a Formula One racing ...
3,15,4,The Wonderful Thing About Tiggers,291,"""The Wonderful Thing About Tiggers"" is the the..."
4,15,7,Winnie the Pooh and the Blustery Day,1652,Winnie the Pooh and the Blustery Day is a 1968...
5,15,9,Little Black Rain Cloud,388,"""Little Black Rain Cloud"" is a song from the 1..."
6,31,1,Gerald Ford,15318,Gerald Rudolph Ford Jr. (born Leslie Lynch Kin...
7,37,4,Carrefour,4743,Carrefour S.A. (] ) is a French multinational ...
8,38,3,Indianapolis Motor Speedway,7651,The Indianapolis Motor Speedway is an automobi...
9,40,9,List of songs recorded by Ellie Goulding,437,English singer and songwriter Ellie Goulding h...


In [13]:
# Cell 8 — Build unique HotpotQA chunks from docs
# Build chunks from docs after per-dataset title+doc deduplication

all_chunks = []
chunk_counter = 1

title_doc_mismatch_count = 0
empty_doc_count = 0
duplicate_doc_count = 0
processed_doc_count = 0

hotpot_seen_doc_keys = set()

for example_idx, example in enumerate(tqdm(examples, desc="Chunking unique HotpotQA docs")):
    titles = example.get("titles", [])
    docs = example.get("docs", [])

    if not isinstance(docs, list):
        continue

    if not isinstance(titles, list):
        titles = []

    if len(titles) != len(docs):
        title_doc_mismatch_count += 1

    for doc_idx, doc_text in enumerate(docs):
        if not isinstance(doc_text, str) or not doc_text.strip():
            empty_doc_count += 1
            continue

        title = titles[doc_idx] if doc_idx < len(titles) else f"untitled_doc_{doc_idx + 1}"

        # Deduplicate only inside HotpotQA
        doc_key = make_doc_key(title, doc_text)

        if doc_key in hotpot_seen_doc_keys:
            duplicate_doc_count += 1
            continue

        hotpot_seen_doc_keys.add(doc_key)
        processed_doc_count += 1

        paragraphs = split_non_empty_paragraphs(doc_text)

        if not paragraphs:
            empty_doc_count += 1
            continue

        units = merge_short_paragraphs_with_next(paragraphs)
        title_chunks = chunk_units_for_one_title(units)

        for ch in title_chunks:
            chunk_id = f"hotpotqa_chunk_{chunk_counter:08d}"

            all_chunks.append({
                "Chunk_id": chunk_id,
                "Title": title,
                "Paragraph_id": ch["paragraph_ids"],
                "Text": ch["text"],
                "Token_count": ch["token_count"]
            })

            chunk_counter += 1

print("Total HotpotQA chunks:", len(all_chunks))
print("Processed unique HotpotQA docs:", processed_doc_count)
print("Skipped duplicate HotpotQA docs:", duplicate_doc_count)
print("Title/docs mismatch examples:", title_doc_mismatch_count)
print("Empty docs skipped:", empty_doc_count)

Chunking unique HotpotQA docs:   0%|          | 0/1000 [00:00<?, ?it/s]

Total HotpotQA chunks: 35029
Processed unique HotpotQA docs: 9810
Skipped duplicate HotpotQA docs: 190
Title/docs mismatch examples: 0
Empty docs skipped: 0


In [14]:
# Cell 9 — Validate HotpotQA chunks
# Validate chunk structure, token counts, and per-dataset deduplication

chunk_ids = [c["Chunk_id"] for c in all_chunks]
token_counts = [c["Token_count"] for c in all_chunks]

assert len(chunk_ids) == len(set(chunk_ids)), "Chunk IDs are not unique."

required_keys = {"Chunk_id", "Title", "Paragraph_id", "Text", "Token_count"}

for c in all_chunks:
    assert set(c.keys()) == required_keys, f"Unexpected keys found: {c.keys()}"
    assert isinstance(c["Chunk_id"], str), "Chunk_id must be a string."
    assert isinstance(c["Title"], str), "Title must be a single string."
    assert isinstance(c["Paragraph_id"], list), "Paragraph_id must be a list."
    assert isinstance(c["Text"], str), "Text must be a string."
    assert isinstance(c["Token_count"], int), "Token_count must be an integer."
    assert c["Token_count"] == count_tokens(c["Text"]), "Token_count mismatch."

over_512 = sum(tc > MAX_CHUNK_TOKENS for tc in token_counts)

assert processed_doc_count == len(hotpot_seen_doc_keys), "HotpotQA processed doc count mismatch."

print("Validation passed.")
print("Chunks over 512 tokens:", over_512)
print("Min tokens:", min(token_counts))
print("Max tokens:", max(token_counts))

print("Unique HotpotQA docs processed:", processed_doc_count)
print("Skipped duplicate HotpotQA docs:", duplicate_doc_count)
print("Unique title+doc pairs inside HotpotQA:", len(hotpot_seen_doc_keys))

Validation passed.
Chunks over 512 tokens: 0
Min tokens: 11
Max tokens: 512
Unique HotpotQA docs processed: 9810
Skipped duplicate HotpotQA docs: 190
Unique title+doc pairs inside HotpotQA: 9810


In [15]:
# Cell 10 — Save chunks as JSON
# Save the final chunks into the idea_1 folder
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Saved JSON file:")
print(OUTPUT_PATH)

Saved JSON file:
/content/drive/MyDrive/final_project/idea_1/hotpotqa_docs_chunks.json


In [16]:
# Cell 11 — Show token statistics
# Show detailed token distribution after chunking
token_series = pd.Series([c["Token_count"] for c in all_chunks], name="Token_count")

stats = token_series.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
display(stats.to_frame())

bucket_summary = pd.DataFrame({
    "bucket": [
        "< 400",
        "400 - 449",
        "450 - 499",
        "500 - 512",
        "> 512"
    ],
    "count": [
        int((token_series < 400).sum()),
        int(((token_series >= 400) & (token_series <= 449)).sum()),
        int(((token_series >= 450) & (token_series <= 499)).sum()),
        int(((token_series >= 500) & (token_series <= 512)).sum()),
        int((token_series > 512).sum()),
    ]
})

bucket_summary["percent"] = round(100 * bucket_summary["count"] / len(token_series), 2)

display(bucket_summary)

summary = pd.DataFrame({
    "metric": [
        "total_chunks",
        "total_tokens",
        "mean_tokens",
        "median_tokens",
        "min_tokens",
        "max_tokens",
        "chunks_below_400",
        "chunks_400_to_512",
        "chunks_450_to_512",
        "chunks_500_to_512",
        "percent_below_400",
        "percent_400_to_512",
        "percent_450_to_512",
        "percent_500_to_512",
    ],
    "value": [
        len(all_chunks),
        int(token_series.sum()),
        round(float(token_series.mean()), 2),
        round(float(token_series.median()), 2),
        int(token_series.min()),
        int(token_series.max()),
        int((token_series < 400).sum()),
        int(((token_series >= 400) & (token_series <= 512)).sum()),
        int(((token_series >= 450) & (token_series <= 512)).sum()),
        int(((token_series >= 500) & (token_series <= 512)).sum()),
        round(100 * (token_series < 400).mean(), 2),
        round(100 * ((token_series >= 400) & (token_series <= 512)).mean(), 2),
        round(100 * ((token_series >= 450) & (token_series <= 512)).mean(), 2),
        round(100 * ((token_series >= 500) & (token_series <= 512)).mean(), 2),
    ]
})

display(summary)

,Token_count
count,35029.000000
mean,378.385338
std,129.879976
min,11.000000
25%,301.000000
50%,432.000000
75%,474.000000
90%,500.000000
95%,506.000000
99%,511.000000


,bucket,count,percent
0,< 400,12734,36.35
1,400 - 449,7120,20.33
2,450 - 499,11477,32.76
3,500 - 512,3698,10.56
4,> 512,0,0.00


,metric,value
0,total_chunks,35029.00
1,total_tokens,13254460.00
2,mean_tokens,378.39
3,median_tokens,432.00
4,min_tokens,11.00
5,max_tokens,512.00
6,chunks_below_400,12734.00
7,chunks_400_to_512,22295.00
8,chunks_450_to_512,15175.00
9,chunks_500_to_512,3698.00


In [17]:
# Cell 12 — Show the first 5 saved chunks
# Display the first 5 chunks saved in the output JSON
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

pd.set_option("display.max_colwidth", 500)

first_5_chunks = pd.DataFrame(saved_chunks[:5])
display(first_5_chunks)

,Chunk_id,Title,Paragraph_id,Text,Token_count
0,hotpotqa_chunk_00000001,Meet Corliss Archer,"[1, 2, 3, 4, 5]","Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956. Although it was CBS's answer to NBC's popular ""A Date with Judy"", it was also broadcast by NBC in 1948 as a summer replacement for ""The Bob Hope Show"". From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS. Despite the program's long run, fewer than 24 episodes are known to exist.\nPriscilla Lyon and Janet Waldo successively portrayed 15-year-old Corliss on radio...",411
1,hotpotqa_chunk_00000002,Meet Corliss Archer,"[6, 7, 8, 9]","""Meet Corliss Archer"" was written by F. Hugh Herbert, who first introduced the character and her friends in the magazine story ""A Private Affair,"" the first of a series of stories. ""Kiss and Tell"" was a 1943 play that was adapted for a 1945 film starring Shirley Temple. The 1949 sequel, ""A Kiss for Corliss"", was re-released in 1954.\nLike many other radio shows, ""Meet Corliss Archer"" made the leap to television with live performances in 1951 and 1952, and from 1954 to 1955, as a syndicated t...",406
2,hotpotqa_chunk_00000003,Shirley Temple,[1],"Shirley Temple Black (April 23, 1928 – February 10, 2014) was an American actress, singer, dancer, businesswoman, and diplomat who was Hollywood's number one box-office draw as a child actress from 1935 to 1938. As an adult, she was named United States ambassador to Ghana and to Czechoslovakia and also served as Chief of Protocol of the United States.",88
3,hotpotqa_chunk_00000004,Shirley Temple,"[2, 3, 4, 5, 6]","Temple began her film career at the age of three in 1932. Two years later, she achieved international fame in ""Bright Eyes"", a feature film designed specifically for her talents. She received a special Juvenile Academy Award in February 1935 for her outstanding contribution as a juvenile performer in motion pictures during 1934. Film hits such as ""Curly Top"" and ""Heidi"" followed year after year during the mid-to-late 1930s. Temple capitalized on licensed merchandise that featured her wholeso...",457
4,hotpotqa_chunk_00000005,Shirley Temple,"[7, 8, 9]","While at the dance school, she was spotted by Charles Lamont, who was a casting director for Educational Pictures. Temple hid behind the piano while she was in the studio. Lamont took a liking to the young actress and invited her to audition; he signed her to a contract in 1932. Educational Pictures was going to launch their ""Baby Burlesks"", multiple short films satirizing recent film and political events using preschool children in every role.\n""Baby Burlesks"" was a series of one-reelers, a...",482


In [18]:
# Cell 15 — Inspect docs paragraph structure
# Check how docs are split into paragraphs using newline characters

sample_example = twowiki_examples[0]

sample_titles = sample_example.get("titles", [])
sample_docs = sample_example.get("docs", [])

print("Number of titles in first example:", len(sample_titles))
print("Number of docs in first example:", len(sample_docs))

if sample_docs:
    sample_doc_text = sample_docs[0]
    sample_doc_title = sample_titles[0] if sample_titles else "untitled_doc_1"

    sample_paragraphs = split_non_empty_paragraphs(sample_doc_text)

    print("Sample title:", sample_doc_title)
    print("Number of non-empty paragraphs in this doc:", len(sample_paragraphs))
    print("\nFirst few paragraphs:")

    for p in sample_paragraphs[:5]:
        print("-" * 80)
        print("Paragraph ID:", p["paragraph_ids"])
        print("Token count:", p["token_count"])
        print(p["text"][:1000])

Number of titles in first example: 10
Number of docs in first example: 10
Sample title: Calloway County High School
Number of non-empty paragraphs in this doc: 4

First few paragraphs:
--------------------------------------------------------------------------------
Paragraph ID: [1]
Token count: 63
Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.
--------------------------------------------------------------------------------
Paragraph ID: [2]
Token count: 7
Organizations: Clubs/Organizations
--------------------------------------------------------------------------------
Paragraph ID: [3]
Token count: 60
State champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (

In [19]:
# Cell 15.5 — Analyze 2WikiMultihopQA duplicate docs
# Check duplicate title+doc pairs inside 2WikiMultihopQA only

twowiki_doc_records = collect_doc_keys_from_examples(twowiki_examples, "2wikimultihopqa")
twowiki_doc_keys = [r["doc_key"] for r in twowiki_doc_records]

twowiki_key_counter = Counter(twowiki_doc_keys)
twowiki_unique_keys = set(twowiki_doc_keys)

twowiki_internal_duplicate_count = sum(
    count - 1 for count in twowiki_key_counter.values() if count > 1
)

print("2WikiMultihopQA total docs:", len(twowiki_doc_keys))
print("2WikiMultihopQA unique title+doc pairs:", len(twowiki_unique_keys))
print("2WikiMultihopQA duplicate docs inside 2WikiMultihopQA:", twowiki_internal_duplicate_count)

twowiki_duplicate_samples = []

for r in twowiki_doc_records:
    if twowiki_key_counter[r["doc_key"]] > 1:
        twowiki_duplicate_samples.append({
            "example_idx": r["example_idx"],
            "doc_idx": r["doc_idx"],
            "title": r["title"],
            "doc_token_count": r["doc_token_count"],
            "text_preview": r["text_preview"]
        })

    if len(twowiki_duplicate_samples) >= 10:
        break

if twowiki_duplicate_samples:
    print("\nSample duplicate title+doc pairs inside 2WikiMultihopQA:")
    display(pd.DataFrame(twowiki_duplicate_samples))
else:
    print("\nNo duplicate title+doc pairs found inside 2WikiMultihopQA.")

2WikiMultihopQA total docs: 10000
2WikiMultihopQA unique title+doc pairs: 5830
2WikiMultihopQA duplicate docs inside 2WikiMultihopQA: 4170

Sample duplicate title+doc pairs inside 2WikiMultihopQA:


,example_idx,doc_idx,title,doc_token_count,text_preview
0,0,0,Calloway County High School,153,"Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.\nOrganizations: Clubs/Organizations\nState champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball..."
1,0,5,Ottawa High School and Junior High School,262,"The Ottawa High School and Junior High School, located at 526 and 506 S. Main St. respectively, are the historic former high school and junior high school in Ottawa, Kansas. The high school was built in 1917, while the junior high school was built from 1927 to 1928; an enclosed hallway connecting the two buildings was built with the junior high school. The high school was the first school in Ottawa to be built solely as a high school and the eighth school built in Ottawa. George P. Washburn ..."
2,0,6,East High School (Denver),3151,"Denver East High School is a public high school located in the City Park neighborhood on the east side of Denver, Colorado, United States. It is part of the Denver Public Schools system, and is one of four original high schools in Denver. The other three are North High School, West High School, and South High School.\nHistory: Denver East High School opened in 1875 and was the first high school in Denver. The first graduating class was in 1877. In 1889, it moved to 19th and Stout Street beca..."
3,3,0,Wesley Barresi,172,"Wesley Barresi (born 3 May 1984) is a South African born first-class and Netherlands international cricketer. He is a right-handed wicket keeper-batsman and also bowls right-arm offbreak.\nCareer: Wesley became the 100th victim to Indian cricketer Yuvraj Singh, when he was dismissed in the 2011 World Cup game against India.\nIn July 2018, he was named in the Netherlands' One Day International (ODI) squad, for their series against Nepal. Ahead of the ODI matches, the International Cricket Cou..."
4,3,2,Henry Moore (cricketer),352,"Henry Walter Moore (1849 – 20 August 1916) was an English-born first-class cricketer who spent most of his life in New Zealand.\nLife and family: Henry Moore was born in Cranbrook, Kent, in 1849. He was the son of the Reverend Edward Moore and Lady Harriet Janet Sarah Montagu-Scott, who was one of the daughters of the 4th Duke of Buccleuch. One of his brothers, Arthur, became an admiral and was knighted. Their great grandfather was John Moore, Archbishop of Canterbury from 1783 to 1805. One ..."
5,3,4,Wale Adebanwi,435,"Wale Adebanwi (born 1969) is a Nigerian-born first Black Rhodes Professor at St Antony's College, Oxford.\nEducation background: Wale Adebanwi graduated with a first degree in Mass Communication from the University of Lagos, and later earned his M.Sc and Ph.D in Political Science from the University of Ibadan. He also has MPhil and Ph.D in Social Anthropology from the University of Cambridge.\nEarly life: Adebanwi worked as a freelance reporter, writer, journalist and editor for many newspap..."
6,3,5,Greg A. Hill (artist),218,"Greg A. Hill is a Canadian-born First Nations artist and curator. He is Kanyen'kehaka, from Six Nations of the Grand River Territory, Ontario.\nBiography: Hill was born and raised in Fort Erie, Ontario. His work as a multidisciplinary artist focuses primarily on installation, performance and digital imaging and explores issues of his Mohawk and French-Canadian identity through the prism of colonialism, nationalism and concepts of place and community.\nHe has been exhibiting his work since 19..."
7,3,7,Hartley Lobban,623,"Hartley W Lobban (9 May 1926 – 15 October 2004) was a Jamaican-born first-class cricketer who played 17 matches for Worcestershire in the early 1950s.\nLife and c

In [20]:
# Cell 16 — Build unique 2WikiMultihopQA chunks from docs
# Build chunks from docs after per-dataset title+doc deduplication

twowiki_chunks = []
twowiki_chunk_counter = 1

twowiki_title_doc_mismatch_count = 0
twowiki_empty_doc_count = 0
twowiki_duplicate_doc_count = 0
twowiki_processed_doc_count = 0

twowiki_seen_doc_keys = set()

for example_idx, example in enumerate(tqdm(twowiki_examples, desc="Chunking unique 2WikiMultihopQA docs")):
    titles = example.get("titles", [])
    docs = example.get("docs", [])

    if not isinstance(docs, list):
        continue

    if not isinstance(titles, list):
        titles = []

    if len(titles) != len(docs):
        twowiki_title_doc_mismatch_count += 1

    for doc_idx, doc_text in enumerate(docs):
        if not isinstance(doc_text, str) or not doc_text.strip():
            twowiki_empty_doc_count += 1
            continue

        title = titles[doc_idx] if doc_idx < len(titles) else f"untitled_doc_{doc_idx + 1}"

        # Deduplicate only inside 2WikiMultihopQA
        doc_key = make_doc_key(title, doc_text)

        if doc_key in twowiki_seen_doc_keys:
            twowiki_duplicate_doc_count += 1
            continue

        twowiki_seen_doc_keys.add(doc_key)
        twowiki_processed_doc_count += 1

        paragraphs = split_non_empty_paragraphs(doc_text)

        if not paragraphs:
            twowiki_empty_doc_count += 1
            continue

        units = merge_short_paragraphs_with_next(paragraphs)
        title_chunks = chunk_units_for_one_title(units)

        for ch in title_chunks:
            chunk_id = f"2wikimultihopqa_chunk_{twowiki_chunk_counter:08d}"

            twowiki_chunks.append({
                "Chunk_id": chunk_id,
                "Title": title,
                "Paragraph_id": ch["paragraph_ids"],
                "Text": ch["text"],
                "Token_count": ch["token_count"]
            })

            twowiki_chunk_counter += 1

print("Total 2WikiMultihopQA chunks:", len(twowiki_chunks))
print("Processed unique 2WikiMultihopQA docs:", twowiki_processed_doc_count)
print("Skipped duplicate 2WikiMultihopQA docs:", twowiki_duplicate_doc_count)
print("Title/docs mismatch examples:", twowiki_title_doc_mismatch_count)
print("Empty docs skipped:", twowiki_empty_doc_count)

Chunking unique 2WikiMultihopQA docs:   0%|          | 0/1000 [00:00<?, ?it/s]

Total 2WikiMultihopQA chunks: 12685
Processed unique 2WikiMultihopQA docs: 5830
Skipped duplicate 2WikiMultihopQA docs: 4170
Title/docs mismatch examples: 0
Empty docs skipped: 0


In [21]:
# Cell 17 — Validate 2WikiMultihopQA chunks
# Validate chunk structure, token counts, and per-dataset deduplication

twowiki_chunk_ids = [c["Chunk_id"] for c in twowiki_chunks]
twowiki_token_counts = [c["Token_count"] for c in twowiki_chunks]

assert len(twowiki_chunk_ids) == len(set(twowiki_chunk_ids)), "Chunk IDs are not unique."

required_keys = {"Chunk_id", "Title", "Paragraph_id", "Text", "Token_count"}

for c in twowiki_chunks:
    assert set(c.keys()) == required_keys, f"Unexpected keys found: {c.keys()}"
    assert isinstance(c["Chunk_id"], str), "Chunk_id must be a string."
    assert isinstance(c["Title"], str), "Title must be a single string."
    assert isinstance(c["Paragraph_id"], list), "Paragraph_id must be a list."
    assert isinstance(c["Text"], str), "Text must be a string."
    assert isinstance(c["Token_count"], int), "Token_count must be an integer."
    assert c["Token_count"] == count_tokens(c["Text"]), "Token_count mismatch."

twowiki_over_512 = sum(tc > MAX_CHUNK_TOKENS for tc in twowiki_token_counts)

assert twowiki_processed_doc_count == len(twowiki_seen_doc_keys), "2WikiMultihopQA processed doc count mismatch."

print("Validation passed.")
print("Chunks over 512 tokens:", twowiki_over_512)
print("Min tokens:", min(twowiki_token_counts))
print("Max tokens:", max(twowiki_token_counts))

print("Unique 2WikiMultihopQA docs processed:", twowiki_processed_doc_count)
print("Skipped duplicate 2WikiMultihopQA docs:", twowiki_duplicate_doc_count)
print("Unique title+doc pairs inside 2WikiMultihopQA:", len(twowiki_seen_doc_keys))

Validation passed.
Chunks over 512 tokens: 0
Min tokens: 7
Max tokens: 512
Unique 2WikiMultihopQA docs processed: 5830
Skipped duplicate 2WikiMultihopQA docs: 4170
Unique title+doc pairs inside 2WikiMultihopQA: 5830


In [22]:
# Cell 18 — Save 2WikiMultihopQA chunks as JSON
# Save the final chunks into the idea_1 folder

with open(TWOWIKI_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(twowiki_chunks, f, ensure_ascii=False, indent=2)

print("Saved 2WikiMultihopQA chunks JSON file:")
print(TWOWIKI_OUTPUT_PATH)

Saved 2WikiMultihopQA chunks JSON file:
/content/drive/MyDrive/final_project/idea_1/2wikimultihopqa_docs_chunks.json


In [23]:
# Cell 19 — Show 2WikiMultihopQA token statistics
# Show detailed token distribution after chunking

twowiki_token_series = pd.Series(
    [c["Token_count"] for c in twowiki_chunks],
    name="Token_count"
)

twowiki_stats = twowiki_token_series.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

display(twowiki_stats.to_frame())

twowiki_bucket_summary = pd.DataFrame({
    "bucket": [
        "< 400",
        "400 - 449",
        "450 - 499",
        "500 - 512",
        "> 512"
    ],
    "count": [
        int((twowiki_token_series < 400).sum()),
        int(((twowiki_token_series >= 400) & (twowiki_token_series <= 449)).sum()),
        int(((twowiki_token_series >= 450) & (twowiki_token_series <= 499)).sum()),
        int(((twowiki_token_series >= 500) & (twowiki_token_series <= 512)).sum()),
        int((twowiki_token_series > 512).sum()),
    ]
})

twowiki_bucket_summary["percent"] = round(
    100 * twowiki_bucket_summary["count"] / len(twowiki_token_series),
    2
)

display(twowiki_bucket_summary)

twowiki_summary = pd.DataFrame({
    "metric": [
        "total_chunks",
        "total_tokens",
        "mean_tokens",
        "median_tokens",
        "min_tokens",
        "max_tokens",
        "chunks_below_400",
        "chunks_400_to_512",
        "chunks_450_to_512",
        "chunks_500_to_512",
        "percent_below_400",
        "percent_400_to_512",
        "percent_450_to_512",
        "percent_500_to_512",
    ],
    "value": [
        len(twowiki_chunks),
        int(twowiki_token_series.sum()),
        round(float(twowiki_token_series.mean()), 2),
        round(float(twowiki_token_series.median()), 2),
        int(twowiki_token_series.min()),
        int(twowiki_token_series.max()),
        int((twowiki_token_series < 400).sum()),
        int(((twowiki_token_series >= 400) & (twowiki_token_series <= 512)).sum()),
        int(((twowiki_token_series >= 450) & (twowiki_token_series <= 512)).sum()),
        int(((twowiki_token_series >= 500) & (twowiki_token_series <= 512)).sum()),
        round(100 * (twowiki_token_series < 400).mean(), 2),
        round(100 * ((twowiki_token_series >= 400) & (twowiki_token_series <= 512)).mean(), 2),
        round(100 * ((twowiki_token_series >= 450) & (twowiki_token_series <= 512)).mean(), 2),
        round(100 * ((twowiki_token_series >= 500) & (twowiki_token_series <= 512)).mean(), 2),
    ]
})

display(twowiki_summary)

,Token_count
count,12685.000000
mean,332.347734
std,154.288786
min,7.000000
25%,199.000000
50%,399.000000
75%,465.000000
90%,496.000000
95%,505.000000
99%,511.000000


,bucket,count,percent
0,< 400,6352,50.07
1,400 - 449,2022,15.94
2,450 - 499,3229,25.46
3,500 - 512,1082,8.53
4,> 512,0,0.00


,metric,value
0,total_chunks,12685.00
1,total_tokens,4215831.00
2,mean_tokens,332.35
3,median_tokens,399.00
4,min_tokens,7.00
5,max_tokens,512.00
6,chunks_below_400,6352.00
7,chunks_400_to_512,6333.00
8,chunks_450_to_512,4311.00
9,chunks_500_to_512,1082.00


In [24]:
# Cell 20 — Display first 5 chunks as dataframe
# Display the first 5 saved chunks in table format

with open(TWOWIKI_OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved_twowiki_chunks = json.load(f)

pd.set_option("display.max_colwidth", 500)

first_5_twowiki_chunks = pd.DataFrame(saved_twowiki_chunks[:5])
display(first_5_twowiki_chunks)

,Chunk_id,Title,Paragraph_id,Text,Token_count
0,2wikimultihopqa_chunk_00000001,Calloway County High School,"[1, 2, 3, 4]","Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, Kirksey High School, Almo High School, New Concord High School, and Faxon High School.\nOrganizations: Clubs/Organizations\nState champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks (2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball...",153
1,2wikimultihopqa_chunk_00000002,Marion High School (Kansas),"[1, 2, 3, 4, 5]","Marion High School is a public high school in Marion, Kansas, USA. It is one of three schools operated by Marion USD 408, and is the sole high school in the district.\nAcademics: The high school is a member of T.E.E.N., a shared video teaching network, started in 1993, between five area high schools. T.E.E.N. video teaching network The high school has a library for student access.\nSports: The Marion High School mascot is a Warrior. All high school athletic and non-athletic competition is ov...",396
2,2wikimultihopqa_chunk_00000003,North Marion High School (West Virginia),"[1, 2, 3, 4]","North Marion High School is a public Double A (""AA"") high school in the U.S. state of West Virginia, with a current enrollment of 851 students.\nNorth Marion High School is located approximately 4 miles from Farmington, West Virginia on US Route 250 north. While it is closer to the city of Mannington, West Virginia, and is often considered to be located in Rachel, West Virginia, the school mailing address is Farmington. Rachel is a small coal mining community located adjacent to the school, ...",338
3,2wikimultihopqa_chunk_00000004,North Marion High School (West Virginia),"[5, 6, 7, 8, 9]","Nickname & Colors: North Marion students, teams and alumni are known as Huskies. The school colors are black and silver. The mascot and colors were chosen by the students of the various consolidated high schools in an election in the spring of 1979. For the first several years the students enjoyed an unofficial “Husky Lunch” consisting of a Moon pie and an RC Cola until outside vending was closed.\nAthletics & Academics: After completion of the original facility, the building process continu...",470
4,2wikimultihopqa_chunk_00000005,Creswell High School (Oregon),"[1, 2, 3, 4, 5]","Creswell High School (Oregon) is a public high school in Creswell, Oregon, United States.\nAcademics: In 2008, 80% of the school's seniors received their high school diploma. Of 84 students, 67 graduated, 12 dropped out, 2 received a modified diploma, and 3 remained in high school.\nIn 2009, 94.9% of the school's seniors received their high school diploma. Of the senior class: 93 graduated, 5 dropped out, and 0 received a modified diploma.\nState championships: Boys Basketball: 1969, 2000, 2...",187


In [25]:
# Cell 21 — Display first 5 chunks with pprint
# Pretty-print the first 5 saved chunks as JSON-like dictionaries

from pprint import pprint

print("First 5 saved 2WikiMultihopQA chunks:")
pprint(saved_twowiki_chunks[:5], width=120, sort_dicts=False)

First 5 saved 2WikiMultihopQA chunks:
[{'Chunk_id': '2wikimultihopqa_chunk_00000001',
  'Title': 'Calloway County High School',
  'Paragraph_id': [1, 2, 3, 4],
  'Text': 'Calloway County High School is a public high school located in Murray, Kentucky. The school was formed from '
          'the consolidation of six high schools from across the county: Hazel High School, Lynn Grove High School, '
          'Kirksey High School, Almo High School, New Concord High School, and Faxon High School.\n'
          'Organizations: Clubs/Organizations\n'
          'State champions: Wrestling: David Woods 195 lbs (2017) Bass Fishing: Bracken Robertson & Dillon Starks '
          '(2013) Boys Cross Country: 1984 (2A) Fast Pitch Softball: 2004 Girls Golf: 2012 (Individual, Anna Hack)\n'
          'W. Earl Brown, actor: Pookie Jones, 1989 KHSAA Mr. Football winner*',
  'Token_count': 153},
 {'Chunk_id': '2wikimultihopqa_chunk_00000002',
  'Title': 'Marion High School (Kansas)',
  'Paragraph_id': [1, 2